## queremos entrenar el modelo del agente ideador

In [41]:
import pandas as pd
from catalog_cluster_ideador import ClusterRecommender
import importlib, catalog_cluster_ideador as ccr
importlib.reload(ccr)
from catalog_cluster_ideador import ClusterRecommender

df = pd.read_csv("../data/articles.csv", dtype={"article_id": str, "product_code": str})

rec = ClusterRecommender(
    text_cols=["prod_name", "detail_desc"],
    cat_cols=[
        # las que ya tenías
        "garment_group_name", "section_name", "index_name",
        "graphical_appearance_name",
        "perceived_colour_master_name", "perceived_colour_value_name",
        # agrega las del JSON (si existen en tu csv)
        "product_type_name", "product_group_name",
        "colour_group_name", "index_group_name"
    ],
    svd_components=128,
    random_state=42,
    #text_weight=6.0,   # description pesa 6
    #cat_weight=0.5     # demás campos pesan 0.5
)

df["__text_all__"] = (
    df.get("prod_name","").fillna("").astype(str) + " " +
    df.get("detail_desc","").fillna("").astype(str)
)

rec.fit(df)
rec.save("agente_ideador_cluster_with_weights.pkl")

In [5]:
descri = "This is a men's long-sleeved shirt in a light blue colour, made from a cotton or cotton-blend fabric. The shirt features a classic collar, button-front fastening, and cuffed sleeves that can be rolled up. It has a slim fit with a fitted silhouette and a straight hem. The shirt is paired with dark blue trousers and accessorized with a black belt and a watch on the left wrist."


recs = rec.recommend_from_text(descri, k=10, dentro_cluster=True)

recs.to_csv("../data/pruebas/agente_visualizador/query_camisa_azul.csv",index=False)
print(recs.head(10))

       article_id product_code               prod_name garment_group_name  \
27810  0620928007      0620928              PIPER POLO       Jersey Basic   
27812  0620928012      0620928              PIPER POLO       Jersey Basic   
27811  0620928008      0620928              PIPER POLO       Jersey Basic   
27807  0620928003      0620928              PIPER POLO       Jersey Basic   
27809  0620928006      0620928              PIPER POLO       Jersey Basic   
27808  0620928005      0620928              PIPER POLO       Jersey Basic   
78995  0796291001      0796291  Christmas Resort Shirt            Unknown   
27806  0620928001      0620928              PIPER POLO       Jersey Basic   
98605  0880008004      0880008          Fatima t-shirt       Jersey Basic   
98604  0880008003      0880008          Fatima t-shirt       Jersey Basic   

                 section_name  index_name perceived_colour_master_name  \
27810           Men Underwear    Menswear                          Red   
2781

## ahora con json

In [40]:

payload =  {
  "description": "This blue shirt is made of a lightweight material and features a classic collar and button-front closure. The sleeves are rolled up to just above the elbow, giving the shirt a relaxed, casual look.",
  "product_type_name": "Shirt",
  "product_group_name": "Upper Body Garment",
  "graphical_appearance_name": "Solid",
  "colour_group_name": "Blue",
  "index_group_name": "Menswear"
}

# si tu df tiene esas columnas con esos nombres:
# da mejores resultados con false
recs = rec.recommend_from_payload(payload, k=10, dentro_cluster=False)

recs.to_csv("../data/pruebas/agente_visualizador/query_camisa_azul_jsonlive.csv",index=False)
print(recs.head(10))

## el que json2 es cuando tenia mas peso las categoricas que la descripcion

        article_id product_code                   prod_name  \
24154   0608071001      0608071                  Gus worker   
55757   0711483004      0711483                 RODEO SHIRT   
102425  0903866001      0903866        REVERB UTILITY SHIRT   
56740   0714889001      0714889          Kingston CNY shirt   
31276   0631708001      0631708     PE Tora Checked Shirt 2   
41299   0668051006      0668051             Ronny l/s shirt   
34222   0640868001      0640868              Kingston shirt   
96232   0869515001      0869515  Urban oversized long shirt   
34223   0640868005      0640868              Kingston shirt   
55306   0710102003      0710102             Jonas L/S shirt   

          garment_group_name         section_name              index_name  \
24154                 Shirts  Contemporary Casual                Menswear   
55757                 Shirts  Contemporary Street                Menswear   
102425                Shirts   Contemporary Smart                Menswear  

## Probamos el pkl

In [ ]:
from catalog_cluster_ideador import ClusterRecommender
import pandas as pd

rec2 = ClusterRecommender.load("agente_ideador_cluster_with_weights.pkl")


payload =  {
  "description": "This blue shirt is made of a lightweight material and features a classic collar and button-front closure. The sleeves are rolled up to just above the elbow, giving the shirt a relaxed, casual look.",
  "product_type_name": "shirt",
  "product_group_name": "upper body garment",
  "graphical_appearance_name": "solid",
  "colour_group_name": "blue",
  "index_group_name": "menswear"
}




# da mejores resultados con false
top_ids = rec2.recommend_from_payload_ids(
    payload,
    k=10,
    dentro_cluster=False
)

print(top_ids)


['0620928007', '0620928012', '0620928006', '0620928005', '0796291001', '0620928003', '0620928008', '0620928001', '0841808001', '0834983001']
